# Sparsity Ratio Sweep — Deep Validation

**Why**: Top-k neuron sparsity gives +21pp on sort and +8pp on mazes — the strongest single-idea gain on sort.

**Goal**: Ratio sweep (0.25 / 0.5 / 0.75) with 3 seeds to find the optimal sparsity level.

**Hardware**: 1 machine x 8 GPUs. 18 runs, ~6h.

Run from the `paper/` directory.

In [ ]:
import sys; sys.path.insert(0, '.'); sys.path.insert(0, '..')
from exp_runner import make_sparsity, run_all, status, collect, plot_delta_bars
%matplotlib inline

In [ ]:
TASKS = ['sort', 'mazes']
SEEDS = [0, 1, 2]
RATIOS = [0.25, 0.5, 0.75]

exps = make_sparsity(tasks=TASKS, seeds=SEEDS, ratios=RATIOS)
print(f'{len(exps)} experiments')
for e in exps[:6]:
    print(f'  {e.name}')
print('  ...')

## Step 1 — Dry run

In [ ]:
run_all(exps, gpus=8, log_root='logs/deep/03_sparsity', dry_run=True)

## Step 2 — Run training

Uncomment to launch (~6h).

In [ ]:
# done, failed = run_all(exps, gpus=8, log_root='logs/deep/03_sparsity')

In [ ]:
status('logs/deep/03_sparsity')

## Step 3 — Results

Plot ratio sweep curves per task.

In [ ]:
df = collect('logs/deep/03_sparsity')
if df.empty:
    print('No results yet.')
else:
    print(df[['name', 'task', 'best_acc', 'delta']].to_string(index=False))
    plot_delta_bars(df, 'Sparsity ratio sweep vs baseline', 'figures/03_sparsity_delta.png')

In [ ]:
# Ratio sweep curves per task
if not df.empty and 'best_acc' in df and df['best_acc'].notna().any():
    import matplotlib.pyplot as plt
    import numpy as np
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, task in zip(axes, ['sort', 'mazes']):
        sub = df[df['task'] == task].copy()
        sub['ratio'] = sub['name'].str.extract(r'sparsity([0-9]+p?[0-9]*)').iloc[:, 0].str.replace('p', '.').astype(float)
        for r in sorted(sub['ratio'].unique()):
            s = sub[sub['ratio'] == r]
            ax.errorbar([r], [s['best_acc'].mean() * 100],
                        yerr=[s['best_acc'].std(ddof=1) * 100 if len(s) > 1 else 0],
                        fmt='o', capsize=5, markersize=9, color='#9467bd' if task == 'sort' else '#ff7f0e')
        from exp_runner import BASELINE_ACC
        ax.axhline(BASELINE_ACC[task] * 100, color='gray', ls='--', alpha=0.5, label='baseline')
        ax.set_xlabel('topk_neurons ratio')
        ax.set_ylabel('best test acc (%)')
        ax.set_title(task)
        ax.legend()
    fig.suptitle('Sparsity ratio sweep (errorbar = std over 3 seeds)')
    fig.tight_layout()
    fig.savefig('figures/03_sparsity_ratio_curve.png', dpi=150, bbox_inches='tight')
    plt.show()